# 🧠 SDNC Pipeline A100 Haute-RAM — Teacher CPU + Student VRAM

**Stratégie :** Teacher 35B BF16 en RAM CPU (~70 GB) + Student 4B en VRAM (~8 GB)

```
RAM  : 70 (teacher) + 6.5 (circuits) + 3 (OS) = ~79.5 / 83.5 GB
VRAM : 8 (student) + 0.1 (bridges) + 1.5 (cache)  = ~10 / 40 GB
```

**Prérequis :** Runtime A100 avec haute RAM (83.5 GB).

**Variables à remplir dans la Cellule 1 ci-dessous.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 1 — Setup + Vérification hardware
# ═══════════════════════════════════════════════════════════════

# --- Variables configurables (À REMPLIR) ---
GITHUB_REPO = ""          # ex: "https://github.com/user/sdnc"
GITHUB_TOKEN = ""         # depuis Colab Secrets : userdata.get('GITHUB_TOKEN')
GCS_BUCKET = "sdnc-models"
N_TRAIN_STEPS = 500       # steps par cycle

# --- Installation dépendances ---
!pip install -q psutil transformers accelerate bitsandbytes \
    google-cloud-storage gitpython snntorch ncps tqdm

# --- Secrets Colab ---
import os
try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN

# --- Clone ou upload repo ---
import subprocess, sys
SDNC_DIR = '/content/sdnc'

if GITHUB_REPO:
    repo_url = GITHUB_REPO
    if GITHUB_TOKEN:
        repo_url = repo_url.replace('https://', f'https://{GITHUB_TOKEN}@')
    if not os.path.exists(f'{SDNC_DIR}/.git'):
        subprocess.run(['git', 'clone', repo_url, SDNC_DIR], check=True)
    else:
        subprocess.run(['git', '-C', SDNC_DIR, 'pull'], check=True)
elif not os.path.exists(f'{SDNC_DIR}/pipeline'):
    print('\u26a0 GITHUB_REPO non configur\u00e9.')
    print('Uploadez le repo manuellement dans /content/sdnc')
    print('ou utilisez : !cp -r /content/drive/MyDrive/sdnc /content/sdnc')

os.chdir(SDNC_DIR)
sys.path.insert(0, SDNC_DIR)

# --- Vérification hardware ---
import psutil, torch

ram_gb = psutil.virtual_memory().total / 1e9
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU   : {gpu_name}')
print(f'VRAM  : {vram_gb:.1f} GB')
print(f'RAM   : {ram_gb:.1f} GB')

assert ram_gb > 80, f'Besoin 83.5 GB RAM, trouv\u00e9 {ram_gb:.1f}'
assert vram_gb > 38, f'Besoin A100 40 GB, trouv\u00e9 {vram_gb:.1f}'

from brain_hybrid.utils.device import get_device, HW_CONFIG
get_device()
profile = HW_CONFIG.get('hw_type', 'unknown')
print(f'\nProfil : {profile}')
assert profile == 'a100_highram', f'Attendu a100_highram, trouv\u00e9 {profile}'
print('\n\u2713 Cellule 1 PASS \u2014 hardware valid\u00e9')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 2 — Chargement teacher 35B BF16 en RAM CPU
# ═══════════════════════════════════════════════════════════════

import time, torch, psutil
from pipeline.teacher_loader import TeacherLoader

loader = TeacherLoader()

ram_before = psutil.Process().memory_info().rss / 1e9
ram_avail = psutil.virtual_memory().available / 1e9
print(f'RAM process : {ram_before:.1f} GB')
print(f'RAM libre   : {ram_avail:.1f} GB')
print()
print('Chargement Qwen3.5-35B-A3B en BF16 sur CPU RAM...')
print('Cela prend environ 3-5 minutes...')

t0 = time.time()
teacher_model, teacher_tokenizer = loader.load(
    model_name='Qwen/Qwen3.5-35B-A3B',
    dtype=torch.bfloat16,
)
elapsed = time.time() - t0

ram_after = psutil.Process().memory_info().rss / 1e9
delta = ram_after - ram_before

print(f'\n{"="*50}')
print(f'  Avant  : {ram_before:.1f} GB')
print(f'  Apr\u00e8s  : {ram_after:.1f} GB')
print(f'  \u0394 RAM  : {delta:.1f} GB (attendu ~70 GB)')
print(f'  Dur\u00e9e  : {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'{"="*50}')

hidden = teacher_model.config.hidden_size
n_layers = teacher_model.config.num_hidden_layers
print(f'\n  hidden_size : {hidden}')
print(f'  num_layers  : {n_layers}')
print(f'\n\u2713 Cellule 2 PASS \u2014 teacher charg\u00e9 en RAM')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 3 — Chargement student 4B en VRAM + init pipeline
# ═══════════════════════════════════════════════════════════════

import torch, psutil
from pipeline.dual_model_config import DualModelConfig
from pipeline.distillation_engine import DistillationEngine
from brain_hybrid.model import BrainHybridModel

config = DualModelConfig(
    teacher_hidden=hidden,
    gcs_bucket=GCS_BUCKET,
    github_repo=GITHUB_REPO,
)

# Student
vram_before = torch.cuda.memory_allocated(0) / 1e9
print(f'VRAM avant : {vram_before:.1f} GB')

brain_config = config.student_brain_config
student = BrainHybridModel(brain_config)

vram_after = torch.cuda.memory_allocated(0) / 1e9
print(f'VRAM apr\u00e8s : {vram_after:.1f} GB  (\u0394 = {vram_after - vram_before:.1f} GB)')

# Engine
engine = DistillationEngine(
    config=config,
    teacher_loader=loader,
    teacher_model=teacher_model,
    teacher_tokenizer=teacher_tokenizer,
    student_brain=student,
    device=torch.device('cuda'),
)

# Budget
ram_total = psutil.virtual_memory().total / 1e9
ram_used = psutil.Process().memory_info().rss / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_used = torch.cuda.memory_allocated(0) / 1e9

print(f'\nBudget m\u00e9moire :')
print(f'  RAM  : {ram_used:.1f} / {ram_total:.0f} GB ({ram_used/ram_total*100:.0f}%)')
print(f'  VRAM : {vram_used:.1f} / {vram_total:.0f} GB ({vram_used/vram_total*100:.0f}%)')
print(f'\n\u2713 Cellule 3 PASS \u2014 student + engine initialis\u00e9s')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 4 — Test 1 step de distillation (timing détaillé)
# ═══════════════════════════════════════════════════════════════

import time, torch, psutil

prompt = 'Explique le predictive coding dans le cerveau selon Karl Friston.'
print(f'Prompt : "{prompt[:60]}..."\n')

# a) Teacher forward (CPU RAM) + transfert vers VRAM
t0 = time.time()
teacher_reps = loader.get_representations(
    teacher_model, teacher_tokenizer, prompt,
    layers=[16, 32, 48, 64], target_device='cuda'
)
t_teacher = time.time() - t0
print(f'Forward teacher (CPU RAM)       : {t_teacher*1000:.0f} ms')
for i, r in enumerate(teacher_reps):
    print(f'  Layer {[16,32,48,64][i]:2d} : {list(r.shape)} ({r.device})')

# b) Student forward (VRAM)
t0 = time.time()
student_reps = student.llm.get_layer_representations(
    prompt, layers=[8, 16, 24, 36]
)
t_student = time.time() - t0
print(f'\nForward student (VRAM)          : {t_student*1000:.0f} ms')

# c) 1 episode complet de distillation
t0 = time.time()
result = engine.distill_episode(prompt)
t_total = time.time() - t0

print(f'1 step distillation complet     : {t_total*1000:.0f} ms')
print(f'\nR\u00e9sultats :')
print(f'  total_loss     : {result["total_loss"]:.4f}')
print(f'  distill_errors : {["{:.4f}".format(e) for e in result["distill_errors"]]}')
print(f'\nM\u00e9moire apr\u00e8s 1 step :')
print(f'  RAM  : {psutil.Process().memory_info().rss / 1e9:.1f} GB')
print(f'  VRAM : {torch.cuda.memory_allocated(0) / 1e9:.1f} GB')
print(f'\n\u2713 Cellule 4 PASS \u2014 1 step sans OOM')

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 5 — Boucle compl\u00e8te (training + distillation)
# ═══════════════════════════════════════════════════════════════

import torch, psutil
from pipeline.full_loop import SDNCPipeline

# Injecter les objets d\u00e9j\u00e0 charg\u00e9s dans le pipeline
pipeline = SDNCPipeline(config)
pipeline.teacher_loader = loader
pipeline.teacher_model = teacher_model
pipeline.teacher_tokenizer = teacher_tokenizer
pipeline.student = student
pipeline.engine = engine
pipeline._initialized = True

print(f'RAM  : {psutil.Process().memory_info().rss / 1e9:.1f} GB')
print(f'VRAM : {torch.cuda.memory_allocated(0) / 1e9:.1f} GB')
print(f'Steps par cycle : {N_TRAIN_STEPS}')
print()

try:
    pipeline.run_full_cycle(n_train_steps=N_TRAIN_STEPS)
except KeyboardInterrupt:
    pipeline.emergency_save()
    print('\nInterrompu proprement \u2014 checkpoint sauvegard\u00e9')

print(f'\nRAM  : {psutil.Process().memory_info().rss / 1e9:.1f} GB')
print(f'VRAM : {torch.cuda.memory_allocated(0) / 1e9:.1f} GB')
print(pipeline.status())

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Cellule 6 — Benchmark + rapport
# ═══════════════════════════════════════════════════════════════

import gc, json, torch, psutil
from pathlib import Path
from brain_hybrid.eval.benchmark import SDNCBenchmark
from pipeline.teacher_loader import TeacherLoader

# D\u00e9charger teacher pour lib\u00e9rer ~70 GB RAM
print('D\u00e9chargement teacher pour benchmark...')
ram_before = psutil.Process().memory_info().rss / 1e9
TeacherLoader.unload(teacher_model)
teacher_model = None
pipeline.teacher_model = None
gc.collect()
ram_after = psutil.Process().memory_info().rss / 1e9
print(f'RAM lib\u00e9r\u00e9e : {ram_before - ram_after:.1f} GB')
print(f'RAM actuelle : {ram_after:.1f} GB')

# Benchmark
print('\nLancement benchmark SDNC...')
bench = SDNCBenchmark(student)
results = bench.run_full_benchmark(
    pc_steps=30,
    n_distractors=50,
    consistency_steps=100,
    forgetting_steps=200,
)

score = results.get('global_score', '?')
print(f'\nScore global : {score}')

# Sauvegarder rapport
report_dir = Path('checkpoints')
report_dir.mkdir(exist_ok=True)
report_path = report_dir / 'benchmark_a100_full.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=str)
print(f'Rapport : {report_path}')

# Afficher CSV monitoring si disponible
csv_path = report_dir / 'monitoring.csv'
if csv_path.exists():
    print(f'\nCSV monitoring : {csv_path}')
    print(open(csv_path).read()[:500])

# Recharger teacher
print('\nRechargement teacher...')
teacher_model, teacher_tokenizer = loader.load(
    model_name='Qwen/Qwen3.5-35B-A3B',
    dtype=torch.bfloat16,
)
pipeline.teacher_model = teacher_model
pipeline.teacher_tokenizer = teacher_tokenizer
if pipeline.engine:
    pipeline.engine.teacher_model = teacher_model
    pipeline.engine.teacher_tokenizer = teacher_tokenizer
print('\u2713 Teacher recharg\u00e9 \u2014 pipeline pr\u00eat pour le prochain cycle.')